# Run v6 GPU terminal-cache extraction + PRA-AOG

Every executable cell starts with `!`. This notebook checks out `pra-aog-v6-gpu-terminal-cache`, generates GPU terminal caches, and optionally trains hierarchical PRA-AOG on those caches.


In [ ]:
!python -c 'from pathlib import Path; p=Path("/tmp/gpu_terminal_cache_v6.env"); p.write_text("export WORKSPACE=\"/home/dfli/instance_slot_aog\"\nexport REPO_DIR=\"$WORKSPACE/clean_v18_v39_v42\"\nexport PARTIMAGENET_ROOT=\"/home/dfli/full_hyco/PartImageNet\"\nexport CONFIG=\"$REPO_DIR/configs/notebook_stage1.yaml\"\nexport BASE_STAGE1_CKPT=\"$REPO_DIR/runs/stage1_quality_upgrade/checkpoints/stage1_best.pt\"\nexport GPU_CACHE_DIR=\"$REPO_DIR/artifacts/strict_aog_gpu_v6\"\nexport GPU_TRAIN_CACHE=\"$GPU_CACHE_DIR/train_strict_aog_terminals.pt\"\nexport GPU_VAL_CACHE=\"$GPU_CACHE_DIR/val_strict_aog_terminals.pt\"\nexport HIER_BUNDLE_DIR=\"$REPO_DIR/artifacts/hier_pra_aog_gpu_v6\"\nexport HIER_BUNDLE=\"$HIER_BUNDLE_DIR/hier_pra_aog_gpu_v6_bundle.pt\"\nexport HIER_RUN_DIR=\"$REPO_DIR/runs/hier_pra_aog_gpu_v6\"\nexport HIER_BEST_CKPT=\"$HIER_RUN_DIR/checkpoints/strict_aog_best.pt\"\nexport PACKAGE_OUT=\"$WORKSPACE/gpu_terminal_cache_pra_aog_v6_results.tar.gz\"\n", encoding="utf-8"); print(p.read_text())'

## 1. Checkout and install


In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && git -C "$REPO_DIR" fetch origin pra-aog-v6-gpu-terminal-cache && (git -C "$REPO_DIR" switch pra-aog-v6-gpu-terminal-cache || git -C "$REPO_DIR" switch --track -c pra-aog-v6-gpu-terminal-cache origin/pra-aog-v6-gpu-terminal-cache) && git -C "$REPO_DIR" pull --ff-only origin pra-aog-v6-gpu-terminal-cache && cd "$REPO_DIR" && python -m pip install -e ".[dev,vision]" && python -c 'import torch; from partcat_hkg.strict_aog.gpu_terminals import GPUTerminalExtractionConfig; print("torch", torch.__version__, "cuda", torch.cuda.is_available()); print(GPUTerminalExtractionConfig())'

## 2. Tests and dataset config


In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && cd "$REPO_DIR" && pytest -q tests/test_gpu_terminals.py tests/test_hier_pra_aog.py tests/test_strict_aog_core.py && python -c 'from pathlib import Path; import os; p=Path(os.environ["CONFIG"]); p.write_text("extends: stage1_quality_upgrade.yaml\npaths:\n  partimagenet_root: " + repr(os.environ["PARTIMAGENET_ROOT"]) + "\n", encoding="utf-8"); print(p.read_text())'

## 3. Smoke GPU cache


In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && cd "$REPO_DIR" && rm -rf "${GPU_CACHE_DIR}_smoke" && python scripts/cache_strict_aog_terminals_gpu.py --config "$CONFIG" --stage1-ckpt "$BASE_STAGE1_CKPT" --out-dir "${GPU_CACHE_DIR}_smoke" --device auto --splits train,val --batch-size 8 --num-workers 0 --threshold 0.40 --support-gate-mode post --support-component-mode best --cc-mask-size 96 --max-cc-iters 96 --max-terminals 32 --mask-size 64 --shard-size 64 --async-writer --store-images --store-images-splits val --max-batches 2 && find "${GPU_CACHE_DIR}_smoke" -maxdepth 2 -type f | sort | head -n 20

## 4. Full GPU cache extraction


In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && cd "$REPO_DIR" && mkdir -p "$GPU_CACHE_DIR" && python scripts/cache_strict_aog_terminals_gpu.py --config "$CONFIG" --stage1-ckpt "$BASE_STAGE1_CKPT" --out-dir "$GPU_CACHE_DIR" --device auto --splits train,val --batch-size 32 --num-workers 2 --threshold 0.40 --support-gate-mode post --support-component-mode best --cc-mask-size 96 --max-cc-iters 96 --max-components-per-part 4 --max-terminals 32 --mask-size 64 --shard-size 4096 --async-writer --writer-queue-size 4 --store-images --store-images-splits val && ls -lh "$GPU_TRAIN_CACHE" "$GPU_VAL_CACHE"

## 5. Build and train hierarchical PRA-AOG on the GPU cache


In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && cd "$REPO_DIR" && mkdir -p "$HIER_BUNDLE_DIR" && python scripts/build_hier_pra_aog.py --cache "$GPU_TRAIN_CACHE" --out "$HIER_BUNDLE" --num-templates-per-class 4 --max-slots-per-template 14 --max-slots-per-part 4 --min-template-support 2 --min-slot-support 0.10 --required-tau 0.50 --min-role-overlap 0.0 --min-edge-support 0.06 --min-edge-count 2 --min-edge-information-gain 0.02 --max-edges-per-template 24 --relation-var-floor 0.006 --geom-var-floor 0.004 --count-max 6 --motif-min-references 2 --motif-min-utility 0.0 --motif-mdl-penalty 0.01 --motif-shrinkage 0.25 --subpart-grid-size 2 --subpart-min-cell-coverage 0.08 --subpart-min-support 8 --subpart-max-per-part 8 --subpart-score-boost 0.35

In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog.py --bundle "$HIER_BUNDLE" --train-cache "$GPU_TRAIN_CACHE" --val-cache "$GPU_VAL_CACHE" --save-dir "${HIER_RUN_DIR}_smoke" --device auto --batch-size 4 --epochs 1 --lr 2e-4 --weight-decay 1e-4 --assignment gpu_mf --relation-weight 1.35 --count-weight 0.10 --missing-weight 0.30 --edge-start-epoch 1 --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.35 --num-workers 0 --max-train-batches 2 --max-val-batches 2

In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && cd "$REPO_DIR" && python scripts/run_hier_pra_aog.py --bundle "$HIER_BUNDLE" --train-cache "$GPU_TRAIN_CACHE" --val-cache "$GPU_VAL_CACHE" --save-dir "$HIER_RUN_DIR" --device auto --batch-size 16 --epochs 20 --lr 2e-4 --weight-decay 1e-4 --assignment gpu_mf --relation-weight 1.35 --count-weight 0.10 --missing-weight 0.30 --edge-start-epoch 1 --top-k 5 --posterior-tau 0.75 --posterior-logits --subpart-score-weight 0.35 --preload-cache --num-workers 0

## 6. Package outputs


In [ ]:
!. /tmp/gpu_terminal_cache_v6.env && tar -czf "$PACKAGE_OUT" -C "$(dirname "$GPU_CACHE_DIR")" "$(basename "$GPU_CACHE_DIR")" -C "$(dirname "$HIER_BUNDLE_DIR")" "$(basename "$HIER_BUNDLE_DIR")" -C "$(dirname "$HIER_RUN_DIR")" "$(basename "$HIER_RUN_DIR")" && ls -lh "$PACKAGE_OUT"